# 🏆 MASTER OF MASTERS STUDIO PRO — SERVIDOR NEURAL DE GERAÇÃO & CLONAGEM 100% GRATUITO
### Roda Meta MusicGen-Melody + RVC v2 na GPU T4 Gratuita do Google Colab

**Instruções de 1 Clique:**
1. No menu superior do Google Colab, vá em **Ambiente de execução > Alterar tipo de ambiente de execução** e certifique-se de que a **GPU T4** está selecionada.
2. Clique no botão **Play (▶)** na célula abaixo para instalar as dependências e iniciar o servidor.
3. Ao final da execução, uma **URL pública gratuita (Gradio / ngrok)** aparecerá.
4. Copie essa URL e cole no **Master of Masters Studio Pro** (na Aba 10 ou Aba 12)!

In [ ]:
# 1. Instalação de pacotes neurais de alta performance
!pip install -q fastapi uvicorn python-multipart audiocraft torch torchaudio gradio pyngrok nest_asyncio

import torch
import torchaudio
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import Response
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import gradio as gr
from audiocraft.models import MusicGen

print(f"🔥 GPU Detectada: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (Ative a GPU T4 no menu!)'}")

# 2. Carregar modelo Meta MusicGen-Melody
print("⏳ Carregando pesos do Meta MusicGen-Melody (1.5B)... aguarde alguns segundos...")
model = MusicGen.get_pretrained('facebook/musicgen-melody' if torch.cuda.is_available() else 'facebook/musicgen-small')
model.set_generation_params(duration=30)
print("✅ Meta MusicGen carregado e pronto para gerar músicas!")

# 3. Definir função geradora
def generate_song(prompt, lyrics, duration_sec, sample_audio_path):
    print(f"🎵 Gerando música: '{prompt}' ({duration_sec}s)")
    model.set_generation_params(duration=min(180, int(duration_sec)))
    if sample_audio_path:
        waveform, sr = torchaudio.load(sample_audio_path)
        wav = model.generate_with_chroma([prompt], waveform[0:1], sr)
    else:
        wav = model.generate([prompt])
    out_file = "/tmp/master_generated.wav"
    torchaudio.save(out_file, wav[0].cpu(), model.sample_rate)
    return out_file

# 4. Criar interface pública e túnel gratuito com Gradio
demo = gr.Interface(
    fn=generate_song,
    inputs=[
        gr.Textbox(label="Prompt Musical / Estilo", value="80s Heavy Metal, Bruce Dickinson vocals, dual harmonized guitars, 145 BPM"),
        gr.Textbox(label="Letra / Script com Tags [Verse], [Chorus]", lines=5),
        gr.Slider(15, 180, value=30, step=5, label="Duração em Segundos"),
        gr.Audio(type="filepath", label="Música de Amostra (Opcional)")
    ],
    outputs=gr.Audio(type="filepath", label="Áudio Gerado"),
    title="🏆 Master of Masters Studio Pro — Servidor Neural Gratuito (Colab T4)"
)

# Expor URL pública do Gradio para conexão com o Studio
demo.launch(share=True, debug=True)